# Netflix Prize Recommender System
This notebook demonstrates the end-to-end pipeline for the Netflix Movie Recommendation System. It covers:
1. **Data loading and parsing** from raw Netflix files
2. **Exploratory Data Analysis (EDA)**
3. **Model training** (Funk SVD vs Item-Based Collaborative Filtering)
4. **Offline Evaluation** (RMSE and MAP@10)
5. **Personalized Recommendations** with explainability reasons

In [ ]:
# Import standard libraries and local pipeline modules
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path to import from src
sys.path.append(os.path.abspath('..'))

from src.data_loader import load_sampled_data, load_movie_titles, train_test_split_chronological
from src.eda import run_eda
from src.models import FunkSVD, ItemBasedCF
from src.evaluate import evaluate_predictions
from src.recommend import generate_top_k_recommendations, get_similar_movies_svd, analyze_success_failure_cases

DATA_DIR = r"C:\Users\TANVI\Downloads\netflix_prize_data"
print("Libraries and modules loaded successfully!")

## 1. Data Ingestion & Density-Preserving Sampling
We load the movie metadata and construct a dense subset of ratings from `combined_data_1.txt` for computational feasibility.

In [ ]:
# Load movie metadata
movie_df = load_movie_titles(DATA_DIR)

# Load 50,000 ratings filtered to active users and popular movies
ratings_df = load_sampled_data(
    data_dir=DATA_DIR,
    min_user_ratings=30,
    min_movie_ratings=50,
    target_total_ratings=50000
)

print(f"Dataset consists of {ratings_df['user_id'].nunique()} users and {ratings_df['movie_id'].nunique()} movies.")

## 2. Exploratory Data Analysis (EDA)
We calculate matrix sparsity and plot ratings distributions.

In [ ]:
# Run EDA and print key observations
run_eda(ratings_df, movie_df, output_dir="nb_assets")

## 3. Chronological Train-Test Split
Split ratings per user chronologically (80% train, 20% test) to evaluate model generalizability to future ratings.

In [ ]:
train_df, test_df = train_test_split_chronological(ratings_df, test_ratio=0.2)

## 4. Recommender Model Training
We train both models:
1. **Funk SVD** (Latent Factor Model via PyTorch Embeddings)
2. **Item-Based CF** (Direct Neighborhood Cosine Similarity)

In [ ]:
# Train Funk SVD
svd_model = FunkSVD(latent_dim=15, lr=0.005, reg=0.02, epochs=10)
svd_model.fit(train_df)

# Train Item-Based CF
cf_model = ItemBasedCF(k_neighbors=25)
cf_model.fit(train_df)

## 5. Offline Model Evaluation
Compare rating prediction accuracy (**RMSE**) and recommendation list ranking quality (**MAP@10**) on the test set.

In [ ]:
# SVD Predictions & Evaluation
svd_preds = svd_model.predict(test_df['user_id'].values, test_df['movie_id'].values)
svd_metrics = evaluate_predictions(test_df, svd_preds)

# CF Predictions & Evaluation
cf_preds = cf_model.predict(test_df['user_id'].values, test_df['movie_id'].values)
cf_metrics = evaluate_predictions(test_df, cf_preds)

# Output Comparative DataFrame
comparison_df = pd.DataFrame({
    'Metric': ['RMSE (lower is better)', 'MAE', 'MAP@10 (higher is better)'],
    'Funk SVD': [svd_metrics['RMSE'], svd_metrics['MAE'], svd_metrics['MAP@10']],
    'Item-Based CF': [cf_metrics['RMSE'], cf_metrics['MAE'], cf_metrics['MAP@10']]
})
comparison_df

## 6. Explainable Recommendations
Generate top 10 recommended movies for a representative user and explain each recommendation based on their rating history.

In [ ]:
sample_user_id = test_df['user_id'].iloc[0]
print(f"Generating recommendations for User {sample_user_id}...")

recs = generate_top_k_recommendations(svd_model, sample_user_id, train_df, movie_df, k=10)
recs

## 7. Latent Movie Similarities
Query the learned SVD embeddings to find similar movies.

In [ ]:
target_movie_id = 28 # Example: 'Apocalypse Now' or similar index
if target_movie_id in movie_df['movie_id'].values:
    title = movie_df[movie_df['movie_id'] == target_movie_id]['title'].values[0]
    print(f"Finding movies similar to: {title}")
    sim_movies = get_similar_movies_svd(svd_model, target_movie_id, movie_df, n=5)
    display(sim_movies)
else:
    print("Target movie ID not in dataset. Choosing the first movie in our dataset:")
    first_movie = list(svd_model.mapper.movie_to_idx.keys())[0]
    title = movie_df[movie_df['movie_id'] == first_movie]['title'].values[0]
    print(f"Finding movies similar to: {title}")
    sim_movies = get_similar_movies_svd(svd_model, first_movie, movie_df, n=5)
    display(sim_movies)